# Dialforge Cloud Benchmark

One-click benchmark for **Qwen 3 1.7B, 4B and 8B**, **faster-whisper small.en**, **Chatterbox Nano**, and the complete **STT → LLM → TTS pipeline for all three models**.

1. Choose **Runtime → Change runtime type → T4 GPU** and save.
2. Click **Runtime → Run all**.
3. Leave the tab open until the final report appears.

This version uses resumable Ollama downloads and an isolated Python environment so Colab's preinstalled packages cannot interfere with Dialforge. No SIP credentials or real calls are used.

In [ ]:
# ONE-CLICK DIALFORGE BENCHMARK
import pathlib, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

BOOTSTRAP_URL = 'https://raw.githubusercontent.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime/main/benchmarks/dialforge_colab_bootstrap.py'
BOOTSTRAP = pathlib.Path('/content/dialforge_colab_bootstrap.py')
REPORT = pathlib.Path('/content/dialforge-benchmark/dialforge-benchmark-report.html')

print('Downloading current Dialforge bootstrap...')
last_error = None
for attempt in range(1, 6):
    try:
        req = urllib.request.Request(BOOTSTRAP_URL, headers={'User-Agent': 'Dialforge-Colab'})
        with urllib.request.urlopen(req, timeout=60) as response:
            BOOTSTRAP.write_bytes(response.read())
        if BOOTSTRAP.stat().st_size > 1000:
            break
    except Exception as exc:
        last_error = exc
        print(f'Download attempt {attempt}/5 failed: {exc}')
        time.sleep(2 * attempt)
else:
    raise RuntimeError(f'Could not download Dialforge bootstrap: {last_error}')

print('Starting resilient Dialforge benchmark setup...')
result = subprocess.run([sys.executable, str(BOOTSTRAP)])
if result.returncode != 0:
    raise RuntimeError('Dialforge bootstrap stopped. The exact failing component is printed above.')

if not REPORT.exists():
    raise RuntimeError('Benchmark completed without an HTML report.')
print('\n=== BENCHMARK COMPLETE ===')
display(HTML(REPORT.read_text(encoding='utf-8')))


### Result files
The completed run creates:
- `/content/dialforge-benchmark/dialforge-benchmark-report.html`
- `/content/dialforge-benchmark/dialforge-benchmark-report.json`

Those are the only files you need to send back for analysis. The model files stay on Google's temporary VM.